<a href="https://colab.research.google.com/github/nlpsoc/emb-diversity/blob/main/examples/colab_demo.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# emb-diversity — Demo

**emb-diversity** is a Python package for measuring the diversity of small- to
medium-sized text (or vector) datasets, using embeddings. All measures work
off vector representations of your data — depending on which embedding model
you pick, you can measure *semantic*, *stylistic*, *speaker*, or other kinds
of diversity.

This notebook demos:

1. The **Python API** — the main way to use the package.
2. The **CLI** — the same functionality from the command line, run here with
   `!` shell commands (that works the same in a local terminal).

📖 Docs: https://nlpsoc.github.io/emb-diversity/
💻 Code: https://github.com/nlpsoc/emb-diversity


## Setup

Install the package from PyPI. The first time we measure diversity, the default embedding model (`all-mpnet-base-v2`, ~420 MB) is downloaded from the Hugging Face Hub and cached — later calls are fast.

In [ ]:
!pip install -q emb-diversity


### Pre-warm the model cache

Run this once, right after install, so every later cell replays from cache
instead of pausing on a live download in the middle of a take. This notebook
touches up to three distinct embedding models: `semantic` and `style` (used
throughout Part 1 and Part 2), plus an explicit `all-MiniLM-L6-v2` model in
section 1.4. If you're cutting section 1.4 for a shorter recording, delete
that line below and only two models will download.

⚠️ On a fresh Colab runtime this cell does the real downloading (~1–2 minutes
total depending on bandwidth) — that wait only has to happen once per
runtime. Run this cell, then use `Edit → Clear all outputs` and re-run the
notebook for your actual take; as long as you don't restart the runtime in
between, the cache stays warm and every cell replays instantly.

In [ ]:
from emb_diversity.embed import resolve_embeddings

_warmup_texts = ["warm up the cache", "so nothing downloads mid-recording"]

# semantic axis -> all-mpnet-base-v2, style axis -> AnnaWegmann/Style-Embedding
for _axis in ("semantic", "style"):
    resolve_embeddings(_warmup_texts, diversity_axis=_axis)

# only needed if you keep section 1.4 (explicit embedding_model) below
resolve_embeddings(_warmup_texts, embedding_model="all-MiniLM-L6-v2")

print("Models cached and ready — later cells will run instantly.")


## Part 1 — Python API

### 1.1 Basic usage

`measure_diversity` is the single entry point: give it a list of texts, get
back a dict of diversity scores. With no other arguments it runs three
default measures (`graph_entropy`, `vendi_score`, `mean_pw_dist`) using
semantic embeddings.


In [ ]:
from emb_diversity import measure_diversity

# more style-diverse, more topic-uniform (all about 80's music)
texts_a = [
    "I thoroughly enjoy the hair bands.",
    "songs of the 80's are the best.",
    "Hip Hop is going DOWNHILL!!!!!",
    "rock music just makes me feel good",
    "The 80's rocked! That generation had the best music!",
]

measure_diversity(texts_a)


Each result carries the score under `value`, the configuration used under
`parameters`, and the installed package `version` — a fingerprint tracing the
number back to the code that produced it.

### 1.2 Comparing two datasets

A diversity score in isolation is hard to interpret — it's unbounded and
depends on dataset size and embedding space. It's much more informative to
**compare two datasets** on the same measure. Let's add a second corpus that
is topically more varied but stylistically more uniform.


In [ ]:
# more style-uniform (formal), more topic-diverse
texts_b = [
    "I thoroughly enjoy the hair bands.",
    "They have not caused any harm to me.",
    "He has a very distinct walk.",
    "It depends on what they will pay.",
    "I would go out with the son of a preacher.",
]

print("texts_a:", measure_diversity(texts_a))
print("texts_b:", measure_diversity(texts_b))


All three default measures agree: `texts_b` is more diverse than `texts_a`
— under **semantic** embeddings.

### 1.3 Diversity axes — semantic vs. style

A **diversity axis** picks the embedding model, i.e. *what kind* of diversity
we measure. Switching from `semantic` to `style` can flip the comparison,
because the two datasets differ in a different way stylistically than they do
topically.


In [ ]:
print("texts_a (style):", measure_diversity(texts_a, diversity_axis="style"))
print("texts_b (style):", measure_diversity(texts_b, diversity_axis="style"))


### 1.4 Choosing a specific embedding model

Instead of an axis, you can also pass any Hugging Face model id directly via
`embedding_model` — this overrides the axis. Here's a smaller, faster model
than the semantic default.


In [ ]:
measure_diversity(texts_a, embedding_model="all-MiniLM-L6-v2")


### 1.5 Running specific measures

`measure_diversity` ships 22 measures in total (`list-measures` on the CLI
shows all of them). Pass `measure=` to run just the ones you want — a single
name, a list of names, or a named set (`variety`, `balance`, `disparity`).


In [ ]:
print(measure_diversity(texts_a, measure=["diameter", "log_determinant"]))
print(measure_diversity(texts_b, measure=["diameter", "log_determinant"]))


### 1.6 Using raw vectors directly

If you're not working with text — or you already have embeddings — pass a
`(n, d)` array of vectors instead of strings. No embedding step runs, so
`parameters["embedding_model"]` comes back `None`.


In [ ]:
import numpy as np

vectors = np.random.RandomState(0).randn(100, 384)
measure_diversity(vectors, measure="mean_pw_dist")


## Part 2 — Command-line interface (CLI)

`emb-diversity` also installs a CLI, for measuring diversity from text/CSV/TSV
files without writing any Python. A notebook can't run a CLI as an interactive
terminal, but a code cell that starts with `!` runs a real shell command — so
every command below works exactly the same way in a local terminal, just
without the `!`.

First, write our texts to a `.txt` file (one text per line) with the
`%%writefile` cell magic.


In [ ]:
%%writefile texts.txt
I thoroughly enjoy the hair bands.
songs of the 80's are the best.
Hip Hop is going DOWNHILL!!!!!
rock music just makes me feel good
The 80's rocked! That generation had the best music!


Run the default measures on the file:

In [ ]:
!emb-diversity measure texts.txt


Use a **named measure set** (`variety`), a different **axis**, and **JSON** output (handy for piping into other tools):

In [ ]:
!emb-diversity measure texts.txt -m variety --axis style --format json


See every available measure, tagged by which set(s) it belongs to:

In [ ]:
!emb-diversity list-measures


See every registered diversity axis and its default model:

In [ ]:
!emb-diversity list-axes


The CLI also reads CSV/TSV directly — pass `--column` if the text column isn't named `text`:

In [ ]:
%%writefile reviews.csv
review_text
"Absolutely loved this product, exceeded expectations!"
"Terrible experience, would not buy again."
"It's fine, does the job, nothing special."
"Best purchase I've made all year!"
"Broke after two days of use."


In [ ]:
!emb-diversity measure reviews.csv --column review_text


## Wrap-up

- **Python API:** `measure_diversity(data, measure=..., diversity_axis=..., embedding_model=...)`
  is the one function you need for most use cases; each of the 22 measures is
  also importable directly (e.g. `from emb_diversity import vendi_score`).
- **CLI:** `emb-diversity measure <file> [OPTIONS]`, plus `list-measures` and
  `list-axes` for discovery.
- Diversity scores are only meaningful **relative** to another dataset on the
  same measure, axis, and embedding model — always compare, don't read a
  single number in isolation.

📖 Full docs: https://nlpsoc.github.io/emb-diversity/
💻 Source: https://github.com/nlpsoc/emb-diversity
🏛️ Built as part of the [DataDivers](https://datadivers-erc.github.io/) project (ERC Starting Grant 101162980).
